# RAG Agéntico & Benchmark Comparativo: Pinecone vs Amazon S3 Vectors

Este notebook implementa un pipeline de **RAG Agéntico (Self-RAG)** utilizando **LangGraph** y **AWS Bedrock (Claude 3 Haiku)**. 

El objetivo principal es comparar el rendimiento, las latencias de búsqueda y la precisión de la información recuperada entre dos bases de datos vectoriales hospedadas en AWS:
1. **Pinecone** (dimensión 1024).
2. **Amazon S3 Vector Buckets** (dimensión 1024).

In [ ]:
import os
import time
import json
import pandas as pd
import boto3
from pinecone import Pinecone
from dotenv import load_dotenv
from langgraph.graph import END, StateGraph
from typing import List, TypedDict, Dict, Any

# Cargar variables de entorno del archivo .env
load_dotenv()

In [ ]:
# 1. Cliente de AWS Bedrock (Claude 3 Haiku para razonamiento veloz y económico)
bedrock_runtime = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")
model_id = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

# 2. Cliente de Pinecone
pinecone_api_key = os.environ.get("PINECONE_API_KEY_AWS")
pc = Pinecone(api_key=pinecone_api_key)
pinecone_index = pc.Index("poc-ia-prospecter")

# 3. Cliente de Amazon S3 Vectors (Ohio)
s3_vectors_client = boto3.client("s3vectors", region_name="us-east-2")
s3_bucket_name = "poc-ia-prospecter"
s3_index_name = "index-data-test"

In [ ]:
# Función para obtener embeddings de Bedrock Titan v2:0 (1024 dimensiones)
def get_embedding(text: str) -> List[float]:
    response = bedrock_runtime.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        body=json.dumps({"inputText": text})
    )
    return json.loads(response["body"].read())["embedding"]

# Función para llamar a Claude 3 Haiku en Bedrock
def llm_call(prompt: str, system_prompt: str = "") -> str:
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1000,
        "system": system_prompt,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0
    })
    response = bedrock_runtime.invoke_model(modelId=model_id, body=body)
    response_body = json.loads(response["body"].read())
    return response_body["content"][0]["text"]

In [ ]:
# Recuperador desde Pinecone
def retrieve_from_pinecone(query: str, k: int = 5) -> List[Dict[str, Any]]:
    query_vector = get_embedding(query)
    res = pinecone_index.query(vector=query_vector, top_k=k, include_metadata=True)
    documents = []
    for match in res.matches:
        documents.append({
            "text": match.metadata.get("text", ""),
            "score": match.score
        })
    return documents

# Recuperador desde Amazon S3 Vectors
def retrieve_from_s3vectors(query: str, k: int = 5) -> List[Dict[str, Any]]:
    query_vector = get_embedding(query)
    res = s3_vectors_client.query_vectors(
        vectorBucketName=s3_bucket_name,
        indexName=s3_index_name,
        queryVector={"float32": query_vector},
        topK=k,
        returnMetadata=True,
        returnDistance=True
    )
    documents = []
    for match in res.get("matches", []):
        documents.append({
            "text": match.get("metadata", {}).get("text", ""),
            "score": match.get("score", 0.0)
        })
    return documents

In [ ]:
# 1. Evaluador de relevancia de documentos (Grader)
def grade_document_relevance(question: str, document_text: str) -> str:
    system = (
        "Eres un evaluador objetivo. Califica la relevancia del documento recuperado para la pregunta del usuario.\n"
        "Si el documento contiene información semántica útil o relacionada con la pregunta directa, evalúalo como 'si'. De lo contrario, califícalo como 'no'.\n"
        "Responde estrictamente con una sola palabra: 'si' o 'no'."
    )
    prompt = f"Pregunta del usuario: {question}\n\nDocumento recuperado:\n{document_text}\n\n¿Es útil o relevante para la pregunta?"
    return llm_call(prompt, system_prompt=system).strip().lower()

# 2. Evaluador de alucinaciones (Hallucination Grader)
def grade_hallucination(documents_text: str, generation: str) -> str:
    system = (
        "Eres un evaluador que verifica la veracidad de la respuesta del asistente.\n"
        "Compara la respuesta generada con los documentos proporcionados. Si la respuesta contiene afirmaciones que NO están explícitamente presentes o deducibles en los documentos, califícalo como 'no' (hay alucinación).\n"
        "Si toda la respuesta está libre de alucinación y respaldada por el contexto, califícalo como 'si'.\n"
        "Responde estrictamente con una sola palabra: 'si' o 'no'."
    )
    prompt = f"Documentos de referencia:\n{documents_text}\n\nRespuesta generada:\n{generation}\n\n¿La respuesta está totalmente respaldada y libre de alucinaciones?"
    return llm_call(prompt, system_prompt=system).strip().lower()

# 3. Evaluador de respuesta (Answer Grader)
def grade_answer(question: str, generation: str) -> str:
    system = (
        "Eres un evaluador de calidad. Comprueba si la respuesta del asistente responde de forma útil a la pregunta planteada.\n"
        "Si la responde útilmente, responde 'si'. Si la evade o no responde lo solicitado, responde 'no'.\n"
        "Responde estrictamente con una sola palabra: 'si' o 'no'."
    )
    prompt = f"Pregunta del usuario: {question}\n\nRespuesta generada:\n{generation}\n\n¿La respuesta responde de forma útil y directa a la pregunta?"
    return llm_call(prompt, system_prompt=system).strip().lower()

# 4. Optimización de consulta (Query Rewriter)
def rewrite_query(question: str) -> str:
    system = (
        "Eres un optimizador de búsquedas. Reescribe la pregunta original para mejorar los resultados en una búsqueda semántica de vectores.\n"
        "Devuelve únicamente la pregunta optimizada reescrita, sin explicaciones ni introducciones."
    )
    prompt = f"Pregunta original: {question}\n\nPregunta optimizada para búsqueda semántica:"
    return llm_call(prompt, system_prompt=system).strip()

In [ ]:
# Definición del Estado del Agente
class AgentState(TypedDict):
    question: str
    query: str
    db_type: str  # 'pinecone' o 's3vectors'
    documents: List[Dict[str, Any]]
    generation: str
    retries: int
    logs: List[str]
    retrieval_latency: float

# Nodos del Grafo
def node_retrieve(state: AgentState) -> Dict[str, Any]:
    question = state["query"]
    db_type = state["db_type"]
    logs = state.get("logs", [])
    
    t0 = time.time()
    if db_type == "pinecone":
        docs = retrieve_from_pinecone(question)
    else:
        docs = retrieve_from_s3vectors(question)
    t1 = time.time()
    
    latency = (t1 - t0) * 1000
    logs.append(f"[{db_type}] Recuperados {len(docs)} documentos en {latency:.2f}ms para: '{question}'")
    return {"documents": docs, "logs": logs, "retrieval_latency": latency}

def node_grade_documents(state: AgentState) -> Dict[str, Any]:
    question = state["question"]
    documents = state["documents"]
    logs = state.get("logs", [])
    
    relevant_docs = []
    for doc in documents:
        relevance = grade_document_relevance(question, doc["text"])
        if relevance == "si":
            relevant_docs.append(doc)
            
    logs.append(f"Evaluador de Docs: {len(relevant_docs)} / {len(documents)} calificados como útiles.")
    return {"documents": relevant_docs, "logs": logs}

def node_generate(state: AgentState) -> Dict[str, Any]:
    question = state["question"]
    documents = state["documents"]
    logs = state.get("logs", [])
    
    if not documents:
        generation = "Lo siento, no he podido encontrar información relevante en el repositorio para responder a tu pregunta."
        logs.append("Sin documentos válidos. Respuesta por defecto generada.")
        return {"generation": generation, "logs": logs}
        
    context = "\n\n".join([d["text"] for d in documents])
    system = (
        "Eres un asistente RAG formal. Responde a la pregunta del usuario utilizando únicamente el contexto proporcionado.\n"
        "Sé conciso, preciso y basa tus respuestas en los hechos. Si la información no está en el contexto, indica amablemente que no cuentas con los detalles."
    )
    prompt = f"Contexto:\n{context}\n\nPregunta: {question}\n\nRespuesta:"
    generation = llm_call(prompt, system_prompt=system)
    logs.append("Respuesta generada a partir del contexto.")
    return {"generation": generation, "logs": logs}

def node_rewrite_query(state: AgentState) -> Dict[str, Any]:
    query = state["query"]
    logs = state.get("logs", [])
    retries = state.get("retries", 0)
    
    new_query = rewrite_query(query)
    logs.append(f"Reescritura de Consulta: '{query}' -> '{new_query}' (Intento {retries + 1})")
    return {"query": new_query, "retries": retries + 1, "logs": logs}

# Lógica Condicional del Grafo
def route_after_grading(state: AgentState) -> str:
    documents = state["documents"]
    retries = state.get("retries", 0)
    if not documents and retries < 2:
        return "rewrite"
    return "generate"

def route_after_generation(state: AgentState) -> str:
    documents = state["documents"]
    generation = state["generation"]
    question = state["question"]
    retries = state.get("retries", 0)
    
    if not documents:
        return "end"
        
    context = "\n\n".join([d["text"] for d in documents])
    
    # 1. Grado de alucinación
    hallucination_grade = grade_hallucination(context, generation)
    if hallucination_grade == "no" and retries < 2:
        return "rewrite"
        
    # 2. Grado de respuesta a la pregunta
    answer_grade = grade_answer(question, generation)
    if answer_grade == "no" and retries < 2:
        return "rewrite"
        
    return "end"

# Ensamblaje de LangGraph
workflow = StateGraph(AgentState)
workflow.add_node("retrieve", node_retrieve)
workflow.add_node("grade_documents", node_grade_documents)
workflow.add_node("generate", node_generate)
workflow.add_node("rewrite_query", node_rewrite_query)

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    route_after_grading,
    {
        "rewrite": "rewrite_query",
        "generate": "generate"
    }
)
workflow.add_edge("rewrite_query", "retrieve")
workflow.add_conditional_edges(
    "generate",
    route_after_generation,
    {
        "rewrite": "rewrite_query",
        "end": END
    }
)
app = workflow.compile()

In [ ]:
# Preguntas de prueba basadas en tus PDFs (Convenios y RTB's Cibertec)
test_questions = [
    "¿Qué convenios tiene Cibertec con la Policía Nacional del Perú (PNP)?",
    "¿Cuáles son los requisitos y el porcentaje de descuento de la Beca Socioeconómica?",
    "¿Qué beneficios de empleabilidad ofrece Cibertec a sus estudiantes y egresados?",
    "¿Qué convenios internacionales tiene Cibertec para la carrera de computación?",
    "¿Qué convenios ofrece Cibertec con el Ministerio de Defensa (MINDEF) o Fuerzas Armadas?"
]

results = []

for db_type in ["pinecone", "s3vectors"]:
    print(f"\n=============================================")
    print(f"Iniciando evaluación para: {db_type.upper()}")
    print(f"=============================================")
    
    for i, question in enumerate(test_questions):
        print(f"\n[Pregunta {i+1}] {question}")
        t_start = time.time()
        
        # Estado inicial
        state = {
            "question": question,
            "query": question,
            "db_type": db_type,
            "documents": [],
            "generation": "",
            "retries": 0,
            "logs": [],
            "retrieval_latency": 0.0
        }
        
        output = app.invoke(state)
        t_end = time.time()
        
        total_latency = (t_end - t_start) * 1000
        print(f"  - Latencia Total: {total_latency:.2f}ms")
        print(f"  - Reescrituras: {output.get('retries', 0)}")
        print(f"  - Respuesta: {output.get('generation', '')[:100]}...")
        
        results.append({
            "Base de Datos": db_type,
            "Pregunta": question,
            "Respuesta": output.get("generation", ""),
            "Latencia Recuperación (ms)": output.get("retrieval_latency", 0.0),
            "Latencia Total (ms)": total_latency,
            "Número Reescrituras": output.get("retries", 0),
            "Logs de Ejecución": "\n".join(output.get("logs", []))
        })

In [ ]:
# Crear el DataFrame comparativo
df_results = pd.DataFrame(results)

# Guardar el benchmark a CSV para persistencia
df_results.to_csv("benchmark_results.csv", index=False)

# Mostrar el reporte resumido
pd.set_option('display.max_columns', None)
display(df_results[["Base de Datos", "Pregunta", "Latencia Recuperación (ms)", "Latencia Total (ms)", "Número Reescrituras"]])

In [ ]:
# Métricas agregadas por Base de Datos
aggregated_df = df_results.groupby("Base de Datos").agg({
    "Latencia Recuperación (ms)": "mean",
    "Latencia Total (ms)": "mean",
    "Número Reescrituras": "mean"
}).reset_index()

print("\n--- REPORTES PROMEDIO POR BASE VECTORIAL ---")
display(aggregated_df)